# Inference Demo - English to German

In [1]:
import torch

from neuralmt.dataset.multi30k import Multi30kDataset
from neuralmt.transformer.nmt import NeuralMachineTranslator
from neuralmt.bpe.bpe import BPETokenizer

/home/rsenn/miniconda3/envs/nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Tokenizer file and nmt weights
TOKENIZER_FILE = "tokenizer_de_en_bpe_8k.pickle"
NMT_WEIGHTS_FILE = "nmt-48.2M-params.pt"

TOKENIZER = BPETokenizer.load(TOKENIZER_FILE)
LANGUAGE_PAIR = ("en", "de")    # German to englisch
PAD_TOKEN = TOKENIZER.pad_id    # 7997
BOS_TOKEN = TOKENIZER.bos_id    # 7998
EOS_TOKEN = TOKENIZER.eos_id    # 7999
PAD_STRING = "<PAD>"
BOS_STRING = "<BOS>"
EOS_STRING = "<EOS>"

# Hyperparameters Transformer
# Some transformer hyperparamters from:
# https://arxiv.org/abs/1706.03762
# NOTE: These hyperparameters max out my RTX 3060 TI (8gb VRAM)
VOCAB_SIZE = len(TOKENIZER)     # 8000
MAX_SEQ_LENGTH = 256
D_MODEL = 512
NUM_LAYERS = 6
NUM_HEADS = 8
DK_DIM = 64
DV_DIM = 64
FF_DIM = 1365
P_DROPOUT = 0.5

nmt = NeuralMachineTranslator(
    NUM_LAYERS,
    NUM_HEADS,
    D_MODEL,
    VOCAB_SIZE,
    MAX_SEQ_LENGTH,
    FF_DIM,
    DK_DIM,
    DV_DIM,
    P_DROPOUT,
)
nmt.load_state_dict(torch.load(NMT_WEIGHTS_FILE, map_location="cpu"))


<All keys matched successfully>

In [3]:
nmt.eval()

NeuralMachineTranslator(
  (embedding): Embedding(8000, 512)
  (embedding_dropout): Dropout(p=0.5, inplace=False)
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (norm_1): RMSNorm((512,), eps=None, elementwise_affine=True)
        (mhd): MultiHeadAttention(
          (q_layer): Linear(in_features=512, out_features=512, bias=False)
          (k_layer): Linear(in_features=512, out_features=512, bias=False)
          (v_layer): Linear(in_features=512, out_features=512, bias=False)
          (out_proj): Linear(in_features=512, out_features=512, bias=False)
          (dropout): Dropout(p=0.5, inplace=False)
          (rope): RoPE()
        )
        (dropout_1): Dropout(p=0.5, inplace=False)
        (norm_2): RMSNorm((512,), eps=None, elementwise_affine=True)
        (ffn): FFNSwiGLU(
          (gate): Linear(in_features=512, out_features=1365, bias=True)
          (down): Linear(in_features=512, out_features=1365, bias=True)
          (up): Linear(in_

In [4]:
# Creating a prompt
prompt = "A man driving his car."
tokens = [BOS_TOKEN] + TOKENIZER.encode(prompt) + [EOS_TOKEN]
print(
    f"Prompt: {prompt}\n"
    f"Tokens: {tokens}"
)

Prompt: A man driving his car.
Tokens: [7998, 65, 326, 3017, 485, 619, 46, 7999]


In [5]:
# Translating the prompt from englisch to german
tokens_tensor = torch.as_tensor(tokens, dtype=torch.int64).unsqueeze(0)
tokens_output_tensor = nmt.generate(tokens_tensor, BOS_TOKEN, EOS_TOKEN, PAD_TOKEN)
tokens_output = tokens_output_tensor.detach().cpu().squeeze(0).tolist()
output = TOKENIZER.decode(tokens_output, skip_special_tokens=True)
print(
    f"Prompt:           {prompt}\n"
    f"Tokens (input):   {tokens}\n"
    f"Tokens (output):  {tokens_output}\n"
    f"Translation:      {output}"
)

Prompt:           A man driving his car.
Tokens (input):   [7998, 65, 326, 3017, 485, 619, 46, 7999]
Tokens (output):  [7998, 277, 325, 728, 524, 1656, 310, 728, 1656, 46, 7999]
Translation:      Ein Mann fährt sein Auto und fährt Auto.


In [6]:
test_set = Multi30kDataset(TOKENIZER, LANGUAGE_PAIR, "test")

n_samples = 32
for i in range(n_samples):
    sample = test_set[i]
    src = sample["src"]
    pred = nmt.generate(src.unsqueeze(0), BOS_TOKEN, EOS_TOKEN, PAD_TOKEN).squeeze(0)
    src_text = TOKENIZER.decode(src.tolist(), True)
    pred_text = TOKENIZER.decode(pred.detach().tolist(), True)    

    print(f"Query       ({LANGUAGE_PAIR[0]}):   {src_text}\n"
          f"Translation ({LANGUAGE_PAIR[1]}):   {pred_text}\n"
    )

Query       (en):   A man in an orange hat starring at something.
Translation (de):   Ein Mann mit einem orangefarbenen Hut starrt auf etwas, das etwas sucht.

Query       (en):   A Boston Terrier is running on lush green grass in front of a white fence.
Translation (de):   Ein Terrier, das vor einem weißen Zaun rennt.

Query       (en):   A girl in karate uniform breaking a stick with a front kick.
Translation (de):   Ein Mann in Karatekleidung macht einen Stock vor einem Fußtritt.

Query       (en):   Five people wearing winter jackets and helmets stand in the snow, with snowmobiles in the background.
Translation (de):   Fünf Personen in Winterjacken und mit Schneekörern im Schnee.

Query       (en):   People are fixing the roof of a house.
Translation (de):   Ein Dachputzer repariert das Dach eines Hauses.

Query       (en):   A man in light colored clothing photographs a group of men wearing dark suits and hats standing around a woman dressed in a strapless gown.
Translation (de): 